# nib — Colab smoke run

**This notebook contains no logic.** It clones, installs, mounts Drive, copies the
data pack to local disk, and calls a script. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository, because the
moment logic moves into a notebook cell the run stops being reproducible.

## What this proves

The two traps the project brief says must work *before* serious training starts:

1. **Data reading is fast enough.** The pack is copied to the VM's local disk
   once, rather than read file-by-file from Drive over the network.
2. **Disconnecting loses nothing.** The run is interrupted deliberately and
   resumed, and the weights must come out bit-identical.

There is no real model here. A dummy network stands in, on purpose: this is a
test of the plumbing, not of anything that learns.

## Before you start

Put `cvl_words_64.lmdb` (about 470 MB, after compaction) in your Drive at
`MyDrive/nib/cvl_words_64.lmdb`.

Runtime → Change runtime type → **T4 is enough**. Do not spend an L4 or an A100
on this.

## 1. What are we running on

Record this. If Colab's Python or torch differs from the local versions, the same
code can behave differently in the two places — which is exactly what the
no-hardcoded-paths and config rules exist to prevent, and what pinning versions
will fix.

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch

print("torch", torch.__version__, "| cuda", torch.version.cuda)

## 2. Clone and install

`torch` is deliberately not in the base dependencies — Colab's build is matched
to its CUDA driver, and installing ours on top can replace a working build with
one that does not match the GPU.

In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track]"

## 3. Mount Drive and copy the pack to local disk

**This copy is the whole point.** Reading 98,179 word images one at a time from
Drive would leave the GPU idle waiting on network round-trips. One sequential
copy of one file takes under a minute, and every read after that is local.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed
!time cp /content/drive/MyDrive/nib/cvl_words_64.lmdb /content/nib/data/processed/
!ls -lh /content/nib/data/processed/

## 4. Is the data all there

One command, and it answers rather than reassures.

In [ ]:
!python scripts/check_data.py

## 5. The exit criterion: interrupt and resume

Trains N steps straight, then trains N/2, kills it, resumes, and trains the rest.
The two sets of weights must be **identical** — not close.

If this passes, phase 1 is done and phase 2 is swapping the dummy model for a
real one inside infrastructure that has already proved itself.

In [ ]:
!python scripts/smoke_train.py --verify-resume --steps 200 --batch-size 16

## 6. Throughput

How many samples per second reach the model. This is the number that says whether
the data pipeline can keep a GPU busy — and if it cannot, no amount of model work
will help.

`--workers 2` matters here: with 0 the main process decodes every image itself
and the GPU waits on it.

In [ ]:
!python scripts/smoke_train.py --steps 300 --batch-size 32 --workers 2

## 7. Checkpoints survive to Drive

A checkpoint on the VM's local disk vanishes with the session. Copying it to
Drive is what makes a multi-day run possible at all.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/checkpoints
!cp -r /content/nib/checkpoints/smoke /content/drive/MyDrive/nib/checkpoints/
!ls -lh /content/drive/MyDrive/nib/checkpoints/smoke/

## What to report back

- Python and torch versions from cell 1, and which GPU you got
- Whether cell 5 printed **PASS**
- The samples/second from cell 6
- Anything that failed, with the error text

The versions matter most: if they differ from the local ones we pin them, before
a difference between the two environments turns into a bug that only appears in
one of them.